In [4]:
from read_raw_data import read_dat_file

# write a function called get reference signal that takes path to a file and gives you a start and stop index for the reference signal
# this function should take the center freq of rx as input and also the sample rate. it should also take tx frequency as input so it knows the bin to look for high power in. 
# when parsing the signal, when the power is high for first time, start index is set. when power is low for first time after start index is set, stop index is set.
# return start and stop index.

iq0_file_path = '/home/ishani/LeoPNT/USRP_data/2_3/utctime_3Feb2024_22:03:48_freq_11_325G_bw_2.0M_ts_1706997828.7235794_inrx0.dat'
iq_path = [iq0_file_path, iq0_file_path.replace('rx0', 'rx1'), iq0_file_path.replace('rx0', 'rx2')]

# iq = [read_dat_file(iq_path[0], None, None), read_dat_file(iq_path[1], None, None), read_dat_file(iq_path[2], None, None)]

# print(iq[0].shape, iq[1].shape, iq[2].shape)

In [7]:
import numpy as np
from matplotlib import pyplot as plt
from utils.calibration import CalibrationSignal
# Set a higher value for agg.path.chunksize
plt.rcParams['agg.path.chunksize'] = 200

file_path = iq0_file_path
center_freq_rx = 11.325e9  # Replace with the actual center frequency of your receiver
sample_rate = 2e6  # Replace with the actual sample rate of your signal
tx_freq = 11.3252e9  # Replace with the actual transmit frequency
nfft = 2**15  # Replace with the fft size you want to use
num_read_samples = nfft  # Replace with the number of samples you want to read at a time
start_from = None
plt.close('all')
plt.figure(figsize=(20, 15))
i = 1
start_index_arr = []
stop_index_arr = []
mean_phase_arr = []
calibration_signal = CalibrationSignal(file_path, center_freq_rx, tx_freq, sample_rate)
while True:
  
    start_index, stop_index = calibration_signal.get_refsigidx(nfft, num_read_samples, start_from)
    print("Start Index:", start_index)
    print("Stop Index:", stop_index)
    if start_index is None or stop_index is None:
        break   
    print("Ref siganl lasts for:", str((stop_index-start_index)/2e6)+" s")
    if start_from is not None:
        print("Capture lasts for:", str((stop_index-start_from)/2e6)+" s")
    start_from = int(stop_index+1e6) # setting for next iteration

    begin_index = int(start_index+2**15)
    end_index = int(stop_index-2**15)
    iq = [read_dat_file(iq_path[0], end_index-begin_index, begin_index), read_dat_file(iq_path[1], end_index-begin_index, begin_index), read_dat_file(iq_path[2], end_index-begin_index, begin_index)]
    iq_ = [iq[0], iq[1], iq[2]]
    # where iq is zero, replace with Nan
    iq_[0] = np.where(iq_[0] == 0, 1e-30, iq_[0])
    iq_[1] = np.where(iq_[1] == 0, 1e-30, iq_[1])
    iq_[2] = np.where(iq_[2] == 0, 1e-30, iq_[2])
    
    del_phi = [(np.angle(iq_[0]/iq_[1])), (np.angle(iq_[0]/iq_[2])), (np.angle(iq_[1]/iq_[2]))]
    start_index_arr.append(begin_index)
    stop_index_arr.append(end_index)
    mean_phase_arr.append([np.mean(del_phi[0]), np.mean(del_phi[1]), np.mean(del_phi[2])])
    print("Mean Phase:", [np.mean(del_phi[0]), np.mean(del_phi[1]), np.mean(del_phi[2])])
    # if i%3 == 0:
    #     plt.subplot(6, 4, int(i/3))
    #     plt.plot(del_phi[0])
    #     plt.plot(del_phi[1])
    #     plt.plot(del_phi[2])
    #     plt.title(str((begin_index-13762560)/2e6)+' s')
    #     plt.ylim([-1.25*np.pi, 1.25*np.pi])
    #     plt.grid()
    # i+=1

# plt.legend(['rx0-rx1', 'rx0-rx2', 'rx1-rx2'], loc="upper right")


# plt.savefig('del_phi.png')

Start Index: 26083328
Stop Index: 28082176
Ref siganl lasts for: 0.999424 s
Mean Phase: [-0.39112452, 1.4083612, 1.799486]
Start Index: 58147392
Stop Index: 60146240
Ref siganl lasts for: 0.999424 s
Capture lasts for: 15.532032 s
Mean Phase: [-0.43075904, 1.3691683, 1.7999281]
Start Index: 90178688
Stop Index: 92210304
Ref siganl lasts for: 1.015808 s
Capture lasts for: 15.532032 s
Mean Phase: [-0.4630449, 1.3365566, 1.7996017]
Start Index: 122242752
Stop Index: 124274368
Ref siganl lasts for: 1.015808 s
Capture lasts for: 15.532032 s
Mean Phase: [-0.49338055, 1.3062226, 1.7996025]
Start Index: 154306816
Stop Index: 156305664
Ref siganl lasts for: 0.999424 s
Capture lasts for: 15.515648 s
Mean Phase: [-0.5214618, 1.2818826, 1.8033447]
Start Index: 186370880
Stop Index: 188369728
Ref siganl lasts for: 0.999424 s
Capture lasts for: 15.532032 s
Mean Phase: [-0.5445186, 1.255152, 1.7996708]
Start Index: 218434944
Stop Index: 220433792
Ref siganl lasts for: 0.999424 s
Capture lasts for: 15.

<Figure size 2000x1500 with 0 Axes>

In [3]:
phase_drift_arr = []
for i in range(len(mean_phase_arr)-1):
    phase_drift_arr.append([(mean_phase_arr[i+1][0]-mean_phase_arr[i][0])/((start_index_arr[i+1]-stop_index_arr[i])/2e6), (mean_phase_arr[i+1][1]-mean_phase_arr[i][1])/((start_index_arr[i+1]-stop_index_arr[i])/2e6), (mean_phase_arr[i+1][2]-mean_phase_arr[i][2])/((start_index_arr[i+1]-stop_index_arr[i])/2e6)])

phase_drift_arr = np.array(phase_drift_arr)
print("Phase Drift:", phase_drift_arr)
print("Mean Phase Drift:", np.mean(phase_drift_arr,axis=0))
print("Std Phase Drift:", np.std(phase_drift_arr,axis=0))

Phase Drift: [[-2.63083549e-03 -2.60152252e-03  2.93406581e-05]
 [-2.14538416e-03 -2.16703735e-03 -2.16888304e-05]
 [-2.01579182e-03 -2.01568290e-03  5.54498950e-08]
 [-1.86598799e-03 -1.61737838e-03  2.48669015e-04]
 [-1.53045003e-03 -1.77431062e-03 -2.43864547e-04]
 [-1.61520797e-03 -1.60931294e-03  5.90294793e-06]
 [-1.49540029e-03 -1.52025439e-03 -2.49411419e-05]
 [-1.29471148e-03 -1.36752115e-03 -7.28215550e-05]
 [-1.54515489e-03 -1.42828632e-03  1.16896300e-04]
 [-1.46567539e-03 -1.37423851e-03  9.14764840e-05]
 [-9.33172191e-04 -1.00458520e-03 -7.13971838e-05]
 [-1.16655624e-03 -1.25587987e-03 -8.92801092e-05]
 [-1.25740308e-03 -1.22676868e-03  3.05750547e-05]
 [-8.82726683e-04 -9.53635216e-04 -9.15556981e-05]
 [-9.56906760e-04 -9.31875093e-04  4.57065563e-05]
 [-8.12598408e-04 -6.41555286e-04  1.71031241e-04]
 [-1.06394486e-03 -1.15988506e-03 -9.59520827e-05]
 [-9.64578649e-04 -9.13632078e-04  5.09346893e-05]
 [-6.76472877e-04 -6.43709910e-04  3.28184165e-05]
 [-1.19931918e-03 